# HealthyAir - Google Cloud Storage configuration

This notebook demonstrates an end-to-end data engineering workflow using the UCI Air Quality dataset and Google Cloud Storage (GCS).
1. Installs the required Python libraries.
2. Configures GCP authentication and project settings.
3. Creates a Google Cloud Storage (GCS) bucket if none exists.
4. Downloads the raw UCI Air Quality CSV and stores it in GCS.
5. Establishes chronological train/dev/test datasets and 8-fold cross-validation splits.
6. Timestamp creation, train/evaluation/test partitions and Target-Next-Hour.
7. Stores the resulting datasets back into GCS.

In [20]:
# Import Libraries
import pyarrow
from pathlib import Path
import pandas as pd
from google.cloud import storage
import numpy as np
from sklearn.model_selection import TimeSeriesSplit

print("Google Cloud Storage imported successfully")

Google Cloud Storage imported successfully


In [2]:
# Google Cloud Storage Configuration
# ---------------------------------------------------------------------

PROJECT_ID = "healthy-air-analysis"
BUCKET_NAME = "uci_air_quality_20119102"
LOCATION = "EU"

print(f"Project: {PROJECT_ID}")
print(f"Bucket: gs://{BUCKET_NAME}")

Project: healthy-air-analysis
Bucket: gs://uci_air_quality_20119102


In [3]:
# Data paths
# ---------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

while not (PROJECT_ROOT / ".git").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project Root:", PROJECT_ROOT)

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_FILE = PROJECT_ROOT / "data" / "raw" / "AirQualityUCI.csv"

RAW_GCS_PATH = "data_raw/uci-air-quality_20119102/v1/AirQualityUCI.csv"
PROCESSED_GCS_PATH = "processed/v1/"

print(Path.cwd())
print("Local raw data:", RAW_FILE)
print("Processed directory:", PROCESSED_DIR)

Project Root: /home/chloekc/datahandling/healthyair
/home/chloekc/datahandling/healthyair/notebooks
Local raw data: /home/chloekc/datahandling/healthyair/data/raw/AirQualityUCI.csv
Processed directory: /home/chloekc/datahandling/healthyair/data/processed


In [4]:
# Google Cloud Storage functions
# ---------------------------------------------------------------------

def get_storage_client(project_id=PROJECT_ID):
    """
    Create authenticated GCS client using
     Application Default Credentials.
    """
    return storage.Client(project=project_id)

In [5]:
# GCS - Bucket
# ---------------------------------------------------------------------

def create_gcs_bucket(
    bucket_name=BUCKET_NAME,
    project_id=PROJECT_ID,
    location=LOCATION,
):
    client = get_storage_client(project_id)

    bucket = client.lookup_bucket(bucket_name)

    if bucket:
        print(f"Bucket already exists: gs://{bucket_name}")
        return bucket

    bucket = client.bucket(bucket_name)

    bucket.iam_configuration.uniform_bucket_level_access_enabled = True

    bucket = client.create_bucket(
        bucket,
        location=location
    )

    print(f"Created bucket: gs://{bucket_name}")

    return bucket

In [6]:
# Upload Files
# ---------------------------------------------------------------------

def upload_file_to_gcs(
    local_path,
    destination_blob_name,
    bucket_name=BUCKET_NAME,
    project_id=PROJECT_ID,
):
    local_path = Path(local_path)

    if not local_path.exists():
        raise FileNotFoundError(local_path)

    client = get_storage_client(project_id)

    bucket = client.bucket(bucket_name)

    blob = bucket.blob(destination_blob_name)

    blob.upload_from_filename(str(local_path))

    print(
        f"Uploaded: {local_path} -> "
        f"gs://{bucket_name}/{destination_blob_name}"
    )

In [7]:
# Upload Raw Dataset
# ---------------------------------------------------------------------

def upload_raw_data():
    upload_file_to_gcs(
        local_path=RAW_FILE,
        destination_blob_name=RAW_GCS_PATH
    )

In [8]:
# Bucket Listing
# ---------------------------------------------------------------------

def list_bucket_contents(
    bucket_name=BUCKET_NAME,
    project_id=PROJECT_ID,
):
    client = get_storage_client(project_id)

    blobs = client.list_blobs(bucket_name)

    print(f"\nContents of gs://{bucket_name}\n")

    for blob in blobs:
        print(blob.name)

In [9]:
# Dataset Loading
# ---------------------------------------------------------------------

air_quality = pd.read_csv(RAW_FILE)

print(air_quality.shape)

air_quality.head()

(9471, 17)


,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Unnamed: 15,Unnamed: 16
0,3/10/2004,18:00:00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578,NaN,NaN
1,3/10/2004,19:00:00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255,NaN,NaN
2,3/10/2004,20:00:00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502,NaN,NaN
3,3/10/2004,21:00:00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867,NaN,NaN
4,3/10/2004,22:00:00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888,NaN,NaN


In [10]:
air_quality.dtypes

Date                 str
Time                 str
CO(GT)           float64
PT08.S1(CO)      float64
NMHC(GT)         float64
C6H6(GT)         float64
PT08.S2(NMHC)    float64
NOx(GT)          float64
PT08.S3(NOx)     float64
NO2(GT)          float64
PT08.S4(NO2)     float64
PT08.S5(O3)      float64
T                float64
RH               float64
AH               float64
Unnamed: 15      float64
Unnamed: 16      float64
dtype: object

In [15]:
# Chronological Train / Evaluation / Test Splits + Cross-validation
# ---------------------------------------------------------------------
# Create chronological 70/15/15 train, evaluation and test splits.
# Generate 8 expanding-window CV folds from training data only(data leakage).
# The splits are stored in the `data/processed` folder and uploaded to GCS.

def create_chronological_splits(
    df: pd.DataFrame,
    train_fraction: float = 0.70,
    evaluation_fraction: float = 0.15,
    n_splits: int = 8,
    cv_gap: int = 1,
    save_local: bool = True,
    upload_to_gcs: bool = True,
    bucket_name: str = BUCKET_NAME,
    project_id: str = PROJECT_ID,
):
#    required_columns = {"Datetime", "NO2(GT)"}
    required_columns = {
        "Date",
        "Time",
        "NO2(GT)"
    }
    missing_columns = required_columns.difference(df.columns)

    if missing_columns:
        raise ValueError(
            "Missing required columns: "
            + ", ".join(sorted(missing_columns))
        )

    if not 0 < train_fraction < 1:
        raise ValueError("train_fraction must be between 0 and 1.")

    if not 0 < evaluation_fraction < 1:
        raise ValueError("evaluation_fraction must be between 0 and 1.")

    if train_fraction + evaluation_fraction >= 1:
        raise ValueError(
            "train_fraction + evaluation_fraction must be less than 1."
        )

    if n_splits < 2:
        raise ValueError("n_splits must be at least 2.")

    # Create df_split
    df_split = df.copy()
    
    # Datetime Creation and Validation
    df_split["Datetime"] = pd.to_datetime(
        df_split["Date"] + " " + df_split["Time"],
        errors="coerce"
    )
    valid_datetimes = df_split["Datetime"].notna().sum()
    if valid_datetimes == 0:
        raise ValueError(
            "Datetime parsing failed for all rows. Check Date/Time format."
        )
    df_split["NO2(GT)"] = pd.to_numeric(
    df_split["NO2(GT)"],
    errors="coerce"
    )

    # The UCI dataset uses -200 as a missing-value marker, replace
    df_split["NO2(GT)"] = df_split["NO2(GT)"].replace(-200, np.nan)

    # Drop NANs and ensure chronological order
    df_split = (
        df_split.dropna(subset=["Datetime"])
        .sort_values("Datetime").reset_index(drop=True)
    )
    if df_split["Datetime"].duplicated().any():
        raise ValueError(
            "Duplicate timestamps found. Resolve them during data cleaning "
            "before creating chronological splits."
        )
    # Create next-hour target and its corresponding timestamp
    df_split["Target_Next_Hour"] = df_split["NO2(GT)"].shift(-1)
    df_split["Target_Datetime"] = df_split["Datetime"].shift(-1)

    # Only retain consecutive hourly observations
    print("Rows before next-hour filtering:", len(df_split))

    print(
        (
            df_split["Datetime"].shift(-1)
            - df_split["Datetime"]
        ).value_counts().head(10)
    )
    valid_next_hour = (
        df_split["Target_Datetime"] ==
        df_split["Datetime"] + pd.Timedelta(hours=1)
    )

    df_split.loc[~valid_next_hour, "Target_Next_Hour"] = pd.NA

    # Apply project-defined NO2 categories
    df_split["Target_Next_Hour"] = pd.cut(
        df_split["Target_Next_Hour"],
        bins=[-np.inf, 100, 200, np.inf],
        labels=["GOOD", "MODERATE", "POOR"]
    )

    # Remove rows without a valid next-hour target
    df_split = df_split.dropna(
        subset=["Target_Next_Hour"]
    ).reset_index(drop=True)

    # Determine chronological boundaries
    n = len(df_split)
    train_end = int(n * 0.70)
    eval_end = int(n * 0.85)
    train_boundary = df_split.loc[train_end - 1, "Datetime"]
    eval_boundary = df_split.loc[eval_end - 1, "Datetime"]

    # Training: target must remain within training period
    train_df = df_split[
        (df_split["Datetime"] <= train_boundary) &
        (df_split["Target_Datetime"] <= train_boundary)
    ].copy()

    # Evaluation: features and target must remain in evaluation period
    eval_df = df_split[
        (df_split["Datetime"] > train_boundary) &
        (df_split["Datetime"] <= eval_boundary) &
        (df_split["Target_Datetime"] <= eval_boundary)
    ].copy()

    # Testing: features and target must remain in testing period
    test_df = df_split[
        (df_split["Datetime"] > eval_boundary)
    ].copy()

    # Save processed splits locally
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    
    train_df.to_parquet(
        PROCESSED_DIR / "train.parquet",
        index=False
    )
    eval_df.to_parquet(
        PROCESSED_DIR / "eval.parquet",
        index=False
    )
    test_df.to_parquet(
        PROCESSED_DIR / "test.parquet",
        index=False
    )
    # Generate 8 expanding-window CV folds from training data only
    tscv = TimeSeriesSplit(
        n_splits=n_splits,
        gap=cv_gap
    )

    cv_folds = {}

    for fold_idx, (train_idx, val_idx) in enumerate(
        tscv.split(train_df), start=1
    ):
        cv_folds[fold_idx] = (
            train_df.iloc[train_idx].copy(),
            train_df.iloc[val_idx].copy()
        )

    # Save processed splits locally
    # PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    
    if upload_to_gcs:    
        upload_file_to_gcs(
            PROCESSED_DIR / "train.parquet",
            f"{PROCESSED_GCS_PATH}train.parquet",
            bucket_name,
            project_id
        )   
        upload_file_to_gcs(
            PROCESSED_DIR / "eval.parquet",
            f"{PROCESSED_GCS_PATH}eval.parquet",
            bucket_name,
            project_id
        )  
        upload_file_to_gcs(
            PROCESSED_DIR / "test.parquet",
            f"{PROCESSED_GCS_PATH}test.parquet",
            bucket_name,
            project_id
        )

    # Save individual CV fold datasets
    cv_dir = PROCESSED_DIR / "cross_validation"
    cv_dir.mkdir(parents=True, exist_ok=True)

    for fold, (fold_train, fold_val) in cv_folds.items():
        train_file = (
            cv_dir /
            f"fold_{fold:02d}_train.parquet"
        )
        val_file = (
            cv_dir /
            f"fold_{fold:02d}_validation.parquet"
        )
        fold_train.to_parquet(
            train_file,
            index=False
        )
        fold_val.to_parquet(
            val_file,
            index=False
        )
        if upload_to_gcs:
            upload_file_to_gcs(
                train_file,
                (
                    f"{PROCESSED_GCS_PATH}"
                    f"cross_validation/"
                    f"fold_{fold:02d}_train.parquet"
                ),
                bucket_name,
                project_id
            )
            upload_file_to_gcs(
                val_file,
                (
                    f"{PROCESSED_GCS_PATH}"
                    f"cross_validation/"
                    f"fold_{fold:02d}_validation.parquet"
                ),
                bucket_name,
                project_id
            )
            print(f"\nGenerated {len(cv_folds)} cross-validation folds.")
    return train_df, eval_df, test_df, cv_folds

In [18]:
train_df, eval_df, test_df, cv_folds = create_chronological_splits(
    air_quality,
    upload_to_gcs=True
    )

Rows before next-hour filtering: 9357
Datetime
0 days 01:00:00    9356
Name: count, dtype: int64
Uploaded: /home/chloekc/datahandling/healthyair/data/processed/train.parquet -> gs://uci_air_quality_20119102/processed/v1/train.parquet
Uploaded: /home/chloekc/datahandling/healthyair/data/processed/eval.parquet -> gs://uci_air_quality_20119102/processed/v1/eval.parquet
Uploaded: /home/chloekc/datahandling/healthyair/data/processed/test.parquet -> gs://uci_air_quality_20119102/processed/v1/test.parquet
Uploaded: /home/chloekc/datahandling/healthyair/data/processed/cross_validation/fold_01_train.parquet -> gs://uci_air_quality_20119102/processed/v1/cross_validation/fold_01_train.parquet
Uploaded: /home/chloekc/datahandling/healthyair/data/processed/cross_validation/fold_01_validation.parquet -> gs://uci_air_quality_20119102/processed/v1/cross_validation/fold_01_validation.parquet

Generated 8 cross-validation folds.
Uploaded: /home/chloekc/datahandling/healthyair/data/processed/cross_valida

TypeError: type() takes 1 or 3 arguments

In [12]:
print(PROJECT_ROOT)
print(RAW_DIR)
print(PROCESSED_DIR)

/home/chloekc/datahandling/healthyair
/home/chloekc/datahandling/healthyair/data/raw
/home/chloekc/datahandling/healthyair/data/processed


In [19]:
print(train_df.shape)
print(eval_df.shape)
print(test_df.shape)
print(f"CV folds: {len(cv_folds)}")

(5398, 20)
(1156, 20)
(1158, 20)
CV folds: 8


def analyse_split_data():
    
# Summarise split sizes and category distributions
    split_summary = pd.DataFrame({
        "observations": [
            len(train_df), len(eval_df), len(test_df)
        ],
        "start_date": [
            train_df["Datetime"].min(),
            eval_df["Datetime"].min(),
            test_df["Datetime"].min()
        ],
        "end_date": [
            train_df["Datetime"].max(),
            eval_df["Datetime"].max(),
            test_df["Datetime"].max()
        ]
    }, index=["Training", "Evaluation", "Testing"])

    print("Chronological splits completed:")
    display(split_summary)

    print("\nTarget category distributions:")
    for name, split in [
        ("Training", train_df),
        ("Evaluation", eval_df),
        ("Testing", test_df)
    ]:
        print(f"\n{name}:")
        print(
            split["Target_Next_Hour"]
            .value_counts(normalize=True)
            .mul(100)
            .round(2)
        )

    print(f"\nGenerated {len(cv_folds)} cross-validation folds.")

    return train_df, eval_df, test_df, cv_folds

